# Hahn-echo dephasing

Measure the qubit echo coherence time $T_2^{Echo}$.

In [ ]:
from pathlib import Path

import numpy as np

from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.experiments.cal11_echo import Echo

## Run parameters

In [ ]:
HARDWARE_CONFIG = Path(r"..\config\hw_config.json")
DEVICE_CONFIG = Path(r"..\config\dut_config.json")
FLUX_CONFIG = Path(r"..\config\flux_config.json")
OUTPUT_DIR = Path("data")
QUBITS = ["q0"]
DELAYS = np.arange(start=0.2, stop=250.2, step=0.2) * 1e-6
REPETITIONS = 100
READOUT_AMPLITUDE = None
DRIVE_OUTPUT_ATTENUATION = None
READOUT_OUTPUT_ATTENUATION = None
READOUT_INPUT_ATTENUATION = None
TIMEOUT = 300
CREATE_DUMMY_CONNECTIONS = False
SIMULATE = False
SIMULATION_T2_ECHO = None  # None uses 30 us
SIMULATION_NOISE = None
SIMULATION_SEED = 7
PLOT_RESULTS = True

## Hardware and experiment

In [ ]:
hardware_agent = create_hardware_agent(
    hardware_configuration=HARDWARE_CONFIG, device_configuration=DEVICE_CONFIG,
    output_dir=OUTPUT_DIR, create_dummy_connections=CREATE_DUMMY_CONNECTIONS,
)
experiment = Echo(hardware_agent=hardware_agent, qubits=QUBITS, flux_config=FLUX_CONFIG)

## Measurement or simulation

In [ ]:
if SIMULATE:
    dataset = experiment.simulated_data(
        delays=DELAYS, repetitions=REPETITIONS, t2_echo=SIMULATION_T2_ECHO,
        noise=SIMULATION_NOISE, seed=SIMULATION_SEED,
    )
else:
    dataset = experiment.run_measurement(
        delays=DELAYS, repetitions=REPETITIONS, readout_amplitude=READOUT_AMPLITUDE,
        drive_output_attenuation=DRIVE_OUTPUT_ATTENUATION,
        readout_output_attenuation=READOUT_OUTPUT_ATTENUATION,
        readout_input_attenuation=READOUT_INPUT_ATTENUATION, timeout=TIMEOUT,
    )
dataset

## Analysis

In [ ]:
results = experiment.analysis()
for qubit, result in results.items():
    print(f"{qubit}: T2 echo = {result.t2_echo / 1e-6:.3f} us" if result.success else f"{qubit}: echo fit failed")
if PLOT_RESULTS:
    experiment.plot()

## Result persistence

The fitted value remains in `results[qubit].t2_echo`; the Scheduler device schema has no T2-echo field.